# 00 — Colab Setup & Launch
**Vision-OCR Accessibility Assistant**

Ce notebook prépare l'environnement Colab et lance l'application Gradio en 4 étapes :

| Étape | Action |
|-------|--------|
| 1 | Monter Google Drive |
| 2 | Installer les dépendances |
| 3 | Télécharger le modèle fine-tuné depuis HuggingFace Hub |
| 4 | Lancer `app.py` (interface Gradio) |

> **Runtime recommandé :** GPU T4 (`Exécution → Modifier le type d'exécution → GPU`)

## Étape 1 — Monter Google Drive et cloner le projet

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/vision-ocr-accessibility-assistant')
GITHUB_REPO  = 'https://github.com/ibou224/vision-ocr-accessibility-assistant.git'

# Cloner ou mettre à jour le projet depuis GitHub
if not (PROJECT_ROOT / 'app.py').exists():
    print("Clonage du projet depuis GitHub...")
    subprocess.run([
        'git', 'clone', GITHUB_REPO, str(PROJECT_ROOT)
    ], check=True)
    print("Projet cloné avec succès.")
else:
    print("Projet déjà présent — mise à jour (git pull)...")
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull'], check=True)
    print("Mis à jour.")

print(f"\nFichiers du projet :")
for f in sorted(PROJECT_ROOT.iterdir()):
    print(f"  {f.name}")

In [ ]:
import sys

# Chemins dérivés
MODEL_CACHE  = PROJECT_ROOT / 'models/trocr'
FINETUNE_DIR = PROJECT_ROOT / 'models/trocr-finetuned'

MODEL_CACHE.mkdir(parents=True, exist_ok=True)
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

# Ajouter le projet au sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"MODEL_CACHE  : {MODEL_CACHE}")
print(f"FINETUNE_DIR : {FINETUNE_DIR}")
print(f"app.py found : {(PROJECT_ROOT / 'app.py').exists()}")

## Étape 2 — Installer les dépendances

> Durée estimée : ~3 min

In [3]:
import subprocess, sys

packages = [
    'python-doctr[torch]',
    'transformers',
    'gradio>=4.0.0',
    'gtts',
    'pyttsx3',
    'editdistance',
    'pyyaml',
    'pillow',
    'matplotlib',
    'huggingface_hub',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    print(f'  ✓ {pkg}')

print('\nToutes les dépendances sont installées.')

  ✓ python-doctr[torch]
  ✓ transformers
  ✓ gradio>=4.0.0
  ✓ gtts
  ✓ pyttsx3
  ✓ editdistance
  ✓ pyyaml
  ✓ pillow
  ✓ matplotlib
  ✓ huggingface_hub

Toutes les dépendances sont installées.


## Étape 3 — Vérifier le modèle disponible

Deux cas possibles :

| Situation | Modèle utilisé | Précision |
|-----------|---------------|----------|
| `models/trocr-finetuned/` présent sur Drive | Modèle fine-tuné (CER 0.20) | Meilleure |
| Dossier absent ou vide | Modèle de base `trocr-base-printed` | Standard |

> `app.py` gère automatiquement le fallback — aucune action requise.

In [4]:
model_files = list(FINETUNE_DIR.glob('*')) if FINETUNE_DIR.exists() else []

if model_files:
    print(f'Modèle fine-tuné trouvé ({len(model_files)} fichiers) ✓')
    for f in model_files:
        print(f'  {f.name}')
else:
    print('Modèle fine-tuné absent.')
    print('→ Le pipeline utilisera automatiquement : microsoft/trocr-base-printed')

Modèle fine-tuné trouvé (6 fichiers) ✓
  generation_config.json
  model.safetensors
  config.json
  tokenizer_config.json
  tokenizer.json
  processor_config.json


## Étape 4 — Lancer l'application Gradio

Un lien public `*.gradio.live` sera généré. Il est valide pendant **72 heures**.

> Le chargement du pipeline prend ~1 min (téléchargement des poids docTR au premier lancement).

In [6]:
import os
from pathlib import Path

drive_root = Path('/content/drive/MyDrive')

# Lister le contenu de MyDrive
print("Contenu de MyDrive :")
for f in sorted(drive_root.iterdir()):
    print(f"  {f.name}")


Contenu de MyDrive :
  1  memory des rimes.pdf
  1 Mémento-des-contraires.pdf
  1 associer script capitales (1).pdf
  1 associer script capitales.pdf
  1 autour-de-la-syllabe.pdf
  1 essai d'ecriture (1).pdf
  1 essai d'ecriture.pdf
  1 loto syllabes.pdf
  1 train-phono.pdf
  1.1 SCRIPT CAPITALE.pdf
  1.Les-bases-essentielles-a-connaitre.xlsx
  10  Compter les SYLLABES FRUITS.pdf
  10  memo-alphabet_differentes-couleurs.pdf
  11   Compter les syllabes TABLEAU DE CLASSEMENT.pdf
  11 Discrimation visuelle 8.pdf
  12   eeà.pdf
  13  fluence.pdf
  13  jE COMMENCE PAR UN2.jpg
  14  JEU Phono en 100 mots JOURNAL DE CHRYS.pdf
  14 Discrimation visuelle 11.pdf
  14 jeu de l'oie phono.png
  15  la bonne syllabe-syllabes.pdf
  16 livre syllabe1.pdf
  17  LOTO DES SYLLABES.pdf
  18  support-phono-1.pdf
  19  meli-melo-des-animaux_2020_pions-doubles-dcjdsg.pdf
  2   trouve la rime simple.pdf
  2  initiale-a-b.pdf
  2  lexique-tri-catégories (1).pdf
  2 -lattaque.pdf
  2 charades-des-syllabes.pd

In [ ]:
import os, runpy

app_path = PROJECT_ROOT / 'app.py'

# Vérification finale avant lancement
assert app_path.exists(), (
    f"app.py introuvable à {app_path}\n"
    "→ Relance la cellule Étape 1 pour cloner le projet depuis GitHub."
)

# Définir PROJECT_ROOT comme variable d'environnement pour app.py
os.environ['PROJECT_ROOT'] = str(PROJECT_ROOT)

print(f"Lancement de {app_path} ...")
runpy.run_path(str(app_path), run_name='__main__')

---
## Dépannage

| Erreur | Solution |
|--------|----------|
| `Fine-tuned model not found` | Relancer l'étape 3 avec le bon token HuggingFace |
| `ModuleNotFoundError: src` | Vérifier que `PROJECT_ROOT` pointe bien vers le dossier du projet |
| `CUDA out of memory` | Passer en CPU : dans `app.py`, forcer `device='cpu'` |
| Lien Gradio non généré | Vérifier que `gradio>=4.0.0` est installé (`!pip install gradio -q`) |
| Latence > 2s | Normal sur CPU — passer sur GPU T4 dans les paramètres du Runtime |